In [ ]:
# The classification is about dogs and cats. I gather a dataset of 1000 images
# 500 for dogs and 500 for cats and i split it, 80% training (400 images for each class) and 20% testing (100 images for each class)
# The final results: 75% accuracy and about the loss, it minimized from 0.695 to 0.452
# In other experiments i reached a better loss but the accuracy was worst and also the results with random photos were not accurate enough

from google.colab import drive
drive.mount('/content/drive')     # Connect the drive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")   # Using GPU for better speed
print(device)

cuda:0


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((64,64)),                             # Make all the images 64x64 pixels
    transforms.RandomHorizontalFlip(),                      # Handles directional changes
    transforms.RandomRotation(15),                          # Randomly rotates the images
    transforms.ColorJitter(brightness=0.2, contrast=0.2),   # Handles lighting variations
    transforms.ToTensor(),                                  # Convert the images to Tensors
    transforms.Normalize((0.5, 0.5, 0.5),(0.5, 0.5, 0.5))   # Normalization to (-1,1)
])

test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [ ]:
train_set = torchvision.datasets.ImageFolder(                            # Load the training set
    root = '/content/drive/MyDrive/Internship/dataset_root/train',
    transform = train_transform
)

test_set = torchvision.datasets.ImageFolder(                             # Load the testing set
    root = '/content/drive/MyDrive/Internship/dataset_root/test',
    transform = test_transform
)


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_set,
    batch_size = 16,     # Give to the model packets of 16 images
    shuffle = True       # Shuffle the images, so the model learn and not just find a pattern
)

test_loader = DataLoader(
    test_set,
    batch_size = 16,
)

In [ ]:
class Net(nn.Module):                      # Define our CNN
  def __init__(self):
    super(Net, self).__init__()
    self.conv1 = nn.Conv2d(3, 6, 5)        # The first layer receives 3 RGB channels, search for 6 characteristics and has a 5x5 pixel kernel size
    self.pool = nn.MaxPool2d(2,2)          # Divide the image to 2x2 pixels parts in order to has better vision
    self.conv2 = nn.Conv2d(6, 16, 5)       # The second layer takes the 6 characteristics and makes them 16
    self.fc1 = nn.Linear(16*13*13, 128)    # The first fully connected layer has 16 feature maps of 13x13 pixels and has 128 neurons
    self.fc2 = nn.Linear(128, 64)          # The second fully connected layer takes the 128 neurons and makes them 64
    self.fc3 = nn.Linear(64, 2)            # The third fully connected layer decides the final answer


  def forward(self, x):                    # The way that the image follows
     x = self.pool(F.relu(self.conv1(x)))  # The self.pool shrinks the image in the middle and the F.relu in case that the number was negative, makes it 0
     x = self.pool(F.relu(self.conv2(x)))
     x = torch.flatten(x, 1)               # Takes the squares and returns them as a straight line of numbers

     x = F.relu(self.fc1(x))
     x = F.relu(self.fc2(x))
     x = self.fc3(x)
     return x

net = Net().to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()                                                    # Calculates how far was the prediction from the reality
optimizer = optim.SGD(net.parameters(), lr=0.005, momentum=0.9, weight_decay=1e-4)   # Updates the model's weights to reduce the errors

In [ ]:
for epoch in range(32):                    # Runs 32 times
  net.train()
  running_loss = 0.0

  for i, data in enumerate(train_loader, 0):
    inputs, labels = data[0].to(device), data[1].to(device)

    optimizer.zero_grad()                  # Clears the memory of the optimizer

    outputs = net(inputs)                  # The forward pass
    loss = criterion(outputs, labels)      # Calculates the loss that made
    loss.backward()                        # Goes back in order to find where the problem is
    optimizer.step()                       # Fixes the weights

    running_loss += loss.item()


  avg_loss = running_loss / len(train_loader)
  print(f'[{epoch + 1}] epoch loss: {avg_loss:.3f}')

[1] epoch loss: 0.695
[2] epoch loss: 0.693
[3] epoch loss: 0.692
[4] epoch loss: 0.692
[5] epoch loss: 0.689
[6] epoch loss: 0.687
[7] epoch loss: 0.682
[8] epoch loss: 0.676
[9] epoch loss: 0.673
[10] epoch loss: 0.667
[11] epoch loss: 0.660
[12] epoch loss: 0.656
[13] epoch loss: 0.637
[14] epoch loss: 0.630
[15] epoch loss: 0.616
[16] epoch loss: 0.631
[17] epoch loss: 0.598
[18] epoch loss: 0.603
[19] epoch loss: 0.589
[20] epoch loss: 0.590
[21] epoch loss: 0.583
[22] epoch loss: 0.564
[23] epoch loss: 0.583
[24] epoch loss: 0.554
[25] epoch loss: 0.530
[26] epoch loss: 0.519
[27] epoch loss: 0.518
[28] epoch loss: 0.504
[29] epoch loss: 0.461
[30] epoch loss: 0.464
[31] epoch loss: 0.466
[32] epoch loss: 0.452


In [ ]:
torch.save(net.state_dict(), 'model(3).pth')
print("Saved")

from google.colab import files
files.download('model(3).pth')

Saved


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
correct = 0                                                      # The number of correct answers
total = 0                                                        # All the images
net.eval()                                                       # Informs than now is the evaluation

with torch.no_grad():                                            # Not calculate the gradients
  for data in test_loader:
    images, labels = data[0].to(device), data[1].to(device)
    outputs = net(images)                                        # Makes the prediction
    _,predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

print(f'Accuracy: {100 * correct / total:.2f}%')

Accuracy: 74.87%


In [ ]:
!pip install gradio -q
import gradio as gr
import torch
import torchvision.transforms as transforms
from PIL import Image

MODEL_PATH = 'model(2).pth'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

net = Net().to(device)

try:
    net.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    print("READY")
except FileNotFoundError:
    print(f"ERROR: FILE {MODEL_PATH} NOT FOUND.")

net.eval()


classes = ['Cat', 'Dog']
test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

def predict(img):
    img_t = test_transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = net(img_t)
        _, predicted = torch.max(outputs, 1)

    return classes[predicted.item()]

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="Cat vs Dog Classifier",
    description="Drag your image here:"
)

demo.launch(debug=True)

READY
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://be422a94c3e480edf0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 420, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error